# 04 · Don't Pay Full Price

**Module 09 · Cost Optimization**

Two more levers:

1. **Send less.** Give the model only the handbook lines that matter for this email (the idea from Module 03, used as a cost tool).
2. **Wait for a discount.** The **Batch API** charges half price for work that does not need an answer right now.

In [1]:
import json
import time

import batch
import inbox
import llm
import strategies
import trim

print("Ready")

Ready


## 1. Send less: trim the prompt
The handbook is about 1,170 tokens, and every email carries all of it. But "how much is express shipping?" needs **one paragraph**. We embed the handbook paragraphs once, and for each email we send only the best few:

*From `trim.py`:*

```python
class HandbookIndex:
    def __init__(self):
        """Split the handbook into paragraphs and embed each one (one small call, done once)."""
        self.paragraphs = []
        for paragraph in re.split(r"\n\s*\n", llm.HANDBOOK):
            if len(paragraph.strip()) > 40:        # skip titles and very short lines
                self.paragraphs.append(paragraph.strip())

        self.vectors, self.setup_cost = llm.embed(self.paragraphs)

    def trimmed_prompt(self, email, how_many=3):
        """Return (system text with only the best paragraphs, embedding cost for this email)."""
        vectors, cost = llm.embed([email])
        question_vector = vectors[0]

        scored = []
        for index in range(len(self.paragraphs)):
            scored.append((llm.similarity(question_vector, self.vectors[index]), index))
        scored.sort(reverse=True)    # best match first

        text = SHORT_RULES + "\n\nHandbook lines:"
        for score, index in scored[:how_many]:
            text = text + "\n\n" + self.paragraphs[index]
        return text, cost
```

In [2]:
index = trim.HandbookIndex()
print("Handbook paragraphs:", len(index.paragraphs))

question = "How much does express shipping cost?"
system_text, cost = index.trimmed_prompt(question, how_many=3)

print()
print(system_text)

Handbook paragraphs: 27

You answer customer emails for an online shop. Use ONLY the handbook lines below. Never invent a rule, a price or a time. Be warm and calm. Reply in at most 3 short sentences, with no greeting and no signature. If the handbook lines do not cover the question, or the customer asks for an exception, mentions a lawyer, or wants a refund over $100, say that a colleague will follow up. Never ask for or accept a card number.

Handbook lines:

Shipping costs: Standard shipping is free on orders over $50 and costs $4.99 below that. Express shipping always costs $12.

Delivery times: Standard shipping takes 3 to 5 business days. Express shipping takes 1 to 2 business days.

Replacements: If a claim is approved, we send a replacement using express shipping at no cost.


In [3]:
full = llm.ask(question, llm.SMALL)
short = llm.ask(question, llm.SMALL, system_text=system_text)

print("Full handbook   :", full["prompt_tokens"], "input tokens   reply:", full["text"][:80])
print("Trimmed prompt  :", short["prompt_tokens"], "input tokens   reply:", short["text"][:80])

Full handbook   : 1154 input tokens   reply: Express shipping always costs $12. If you have any other questions, a colleague 
Trimmed prompt  : 190 input tokens   reply: Express shipping always costs $12.


Far fewer tokens, the same answer. Now the **honest question**: does it stay correct over the whole test set, including hard emails that need **two or three** paragraphs at once? We try keeping 3 paragraphs and 6 paragraphs. (About 112 calls, a minute or two.)

In [4]:
def run_trimmed(how_many):
    """Answer every test email with only the best 'how_many' paragraphs. Always the cheap model, so only the prompt differs."""
    total_cost = 0.0
    perfect = 0
    tokens = 0
    misses = []
    for item in inbox.EMAILS:
        system_text, search_cost = index.trimmed_prompt(item["email"], how_many)
        answer = llm.ask(item["email"], llm.SMALL, system_text=system_text)
        total_cost = total_cost + answer["cost"] + search_cost
        tokens = tokens + answer["prompt_tokens"]
        if inbox.score_reply(answer["text"], item["facts"]) == 1.0:
            perfect = perfect + 1
        else:
            misses.append(item["email"][:70])
    return total_cost, perfect, tokens / len(inbox.EMAILS), misses


full_row = strategies.evaluate(strategies.always_small, inbox.EMAILS)
print("Full handbook".ljust(20), "$ per 1,000: " + format(full_row["cost_per_1000"], ".3f"),
      "  all facts: " + str(full_row["perfect"]) + "/28   input tokens per email: about 1170")

for how_many in [3, 6]:
    cost, perfect, avg_tokens, misses = run_trimmed(how_many)
    print(("Best " + str(how_many) + " paragraphs").ljust(20), "$ per 1,000: " + format(cost / len(inbox.EMAILS) * 1000, ".3f"),
          "  all facts: " + str(perfect) + "/28   input tokens per email: about " + str(round(avg_tokens)))
    for email in misses:
        print("      missed:", email)

Full handbook        $ per 1,000: 0.115   all facts: 26/28   input tokens per email: about 1170


Best 3 paragraphs    $ per 1,000: 0.051   all facts: 23/28   input tokens per email: about 230
      missed: I bought headphones 25 days ago and the left side stopped working. Can
      missed: My phone stopped working after 14 months. Is it still under warranty, 
      missed: I am very frustrated. My earbuds arrived broken yesterday. What happen
      missed: It has been 35 days since my order arrived. Please make an exception a
      missed: I ordered three items: $19.99, $14.50 and $9.00. Will I pay for standa


Best 6 paragraphs    $ per 1,000: 0.067   all facts: 25/28   input tokens per email: about 332
      missed: I am very frustrated. My earbuds arrived broken yesterday. What happen
      missed: It has been 35 days since my order arrived. Please make an exception a
      missed: I ordered three items: $19.99, $14.50 and $9.00. Will I pay for standa


This is the trade-off in numbers. A shorter prompt is cheaper and faster, but **if the search misses the paragraph the answer needs, the model has nothing to answer from**. Notice **which** emails fail: usually the ones that need several rules at once, or the one that depends on the "a person must take over" section, which does not *look like* the question.

> **Two warnings.**
> 1. Trimming and prompt caching **work against each other**: a trimmed prompt is under 1,024 tokens, so it is never cached, and it changes for every email. It is cheaper only because it is much shorter. Measure the combination.
> 2. Trimming is the **riskiest** lever in this module. Never ship it without a test set like ours.

## 2. The Batch API: half price, if you can wait
Not everything is urgent. Re-checking yesterday's replies, labelling old tickets, summarising the week: these can wait. Upload a **file of requests**, and OpenAI works through it in the background for **50% less**, within 24 hours (small batches often finish in minutes, but you must not depend on it).

The four steps (all in `batch.py`): **prepare** a `.jsonl` file, **submit** it, **check** the status, **get the results**.

*From `batch.py`:*

```python
def make_batch_file(items, model, path):
    """Write one request per line. The custom_id is how we match answers to questions later."""
    LOG_DIR.mkdir(exist_ok=True)
    with open(path, "w", encoding="utf-8") as file:
        for number in range(len(items)):
            request = {
                "custom_id": "email-" + str(number),
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": model,
                    "messages": [
                        {"role": "system", "content": llm.HANDBOOK},
                        {"role": "user", "content": items[number]["email"]},
                    ],
                    "max_tokens": 150,
                    "temperature": 0,
                },
            }
            file.write(json.dumps(request) + "\n")
```

In [5]:
batch.make_batch_file(inbox.EMAILS[:6], llm.SMALL, batch.INPUT_FILE)

first_line = open(batch.INPUT_FILE, encoding="utf-8").readline()
request = json.loads(first_line)
request["body"]["messages"][0]["content"] = request["body"]["messages"][0]["content"][:60] + "... (the handbook)"
print("One line of the file:")
print(json.dumps(request, indent=2))

One line of the file:
{
  "custom_id": "email-0",
  "method": "POST",
  "url": "/v1/chat/completions",
  "body": {
    "model": "gpt-4o-mini",
    "messages": [
      {
        "role": "system",
        "content": "SUPPORT HANDBOOK - Online Shop (read this fully before you a... (the handbook)"
      },
      {
        "role": "user",
        "content": "How much does express shipping cost?"
      }
    ],
    "max_tokens": 150,
    "temperature": 0
  }
}


Each line is one complete request. The `custom_id` is how we match answers to questions afterwards, because batch answers may come back in any order.

Now the arithmetic. What would a day of 10,000 emails cost, at the normal price and at the batch price?

In [6]:
typical_input = 1170
typical_output = 40
emails = 10000

for model in [llm.SMALL, llm.BIG]:
    normal = llm.cost_of(model, typical_input, 0, typical_output) * emails
    half = llm.cost_of(model, typical_input, 0, typical_output, batch=True) * emails
    print(model.ljust(12), "normal: $" + format(normal, ".2f").rjust(7), "   batch: $" + format(half, ".2f").rjust(7), "   saved: $" + format(normal - half, ".2f"))

gpt-4o-mini  normal: $   1.99    batch: $   1.00    saved: $1.00
gpt-4o       normal: $  33.25    batch: $  16.62    saved: $16.62


Let us run a real (tiny) batch of 6 emails. We submit it, then check every 15 seconds for up to 5 minutes. If it is not done by then, the notebook simply tells you, and the batch carries on in the background. That is how batches behave in real life.

In [7]:
batch_id = batch.submit(batch.INPUT_FILE)
print("Batch started:", batch_id)

started = time.time()
status = None
for attempt in range(20):
    status = batch.check(batch_id)
    print(str(round(time.time() - started)).rjust(4), "s   status:", status.status.ljust(12),
          "done:", status.request_counts.completed, "of", status.request_counts.total)
    if status.status in ["completed", "failed", "expired", "cancelled"]:
        break
    time.sleep(15)

Batch started: batch_6b03b669badc8190af4cca5c76760c32
   0 s   status: validating   done: 0 of 0


  15 s   status: in_progress  done: 5 of 6


  31 s   status: in_progress  done: 5 of 6


  47 s   status: in_progress  done: 5 of 6


  62 s   status: in_progress  done: 5 of 6


  77 s   status: in_progress  done: 5 of 6


  93 s   status: in_progress  done: 5 of 6


 108 s   status: in_progress  done: 5 of 6


 123 s   status: in_progress  done: 5 of 6


 138 s   status: completed    done: 6 of 6


In [8]:
if status.status == "completed":
    answers = batch.results(batch_id)
    total = 0.0
    for custom_id in answers:
        total = total + answers[custom_id]["cost"]
        print(custom_id.ljust(9), answers[custom_id]["text"][:90].replace("\n", " "))
    print()
    print("All 6 answers cost $" + format(total, ".5f"), "(at the batch price)")
else:
    print("Not finished yet. That is normal for a batch. Later you can run:")
    print("    python batch.py check", batch_id)
    print("    python batch.py results", batch_id)

email-0   Express shipping always costs $12. If you have any other questions, a colleague can help.
email-1   Sorry, we do not ship outside the country yet. A colleague can help if you have any other 
email-2   Yes, we accept UPI as a payment method. If you have any trouble paying, a colleague can he
email-3   Standard shipping takes 3 to 5 business days. If you have any other questions, feel free t
email-4   No, standard shipping is free on orders over $50. You won't pay anything for shipping.
email-5   Yes, you can cancel your order for free since it has not shipped yet. Please email support

All 6 answers cost $0.00056 (at the batch price)


## 3. Smaller levers
These are worth knowing, and each is one line of code:

| Lever | What it does | Watch out for |
|---|---|---|
| `max_tokens` | a ceiling on the reply length. Output costs 4 times more than input | a ceiling that is too low **cuts the reply in the middle** |
| "Reply in at most 3 sentences" | shorter replies, and fewer tokens | the model must still be allowed to be complete |
| A shorter, tighter handbook | fewer input tokens for every email | removing a rule removes the behaviour |
| `temperature=0` | not a cost lever, but it makes results repeatable, so your measurements are comparable | |

## What you learned
- **Trimming the prompt** is the biggest saving per call, and the riskiest. Measure quality, not just cost.
- **The Batch API** halves the price of anything that can wait. Use it for overnight work, never for a customer waiting on a reply.
- Small levers add up, but only after you have measured them.

**Next:** all the levers on the same day of traffic, in one table.